# **XGBoost modeling**

## Dự đoán số lượng bán ra (quantity_sold) cho một sản phẩm dựa trên tất cả biến đầu vào.

Các bước chính:
1. Load dữ liệu đã được preprocessing và chia train-test split, trong tập train lại chia thành train-validation.
2. Huấn luyện mô hình XGBoost Regressor trên tập train. Tìm các feature quan trọng.
3. Fine-tune hyperparameters với GridSearchCV hoặc RandomizedSearchCV.
4. Train lại mô hình với các hyperparameters tốt nhất trên toàn bộ tập train (train + valid).
5. Đánh giá mô hình trên tập test trước với RMSE, MAE, R². Với biến mục tiêu `quantity_sold` đã log1p1, cần chuyển ngược về giá trị gốc trước khi tính các metric.

In [ ]:
%pip install xgboost

In [ ]:
# import thư viện cần thiết để train model
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from xgboost import XGBRegressor

# Tải dữ liệu từ file CSV
train_data = pd.read_csv('../data/processed/train_data_final.csv')
test_data = pd.read_csv('../data/processed/test_data_final.csv')

# Tách biến mục tiêu và đặc trưng
y_test = test_data['quantity_sold']
X_test = test_data.drop(columns=['quantity_sold'])

# Tách train thành train và validation
X_train, X_val, y_train, y_val = train_test_split(
    train_data.drop(columns=['quantity_sold']), 
    train_data['quantity_sold'], 
    test_size=0.2, 
    random_state=42
)


In [ ]:
# 2. Huấn luyện mô hình XGBoost Regressor trên tập train. Tìm các feature quan trọng.
xgb_model = XGBRegressor(
    n_estimators=10000,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    early_stopping_rounds=50
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

# Lấy feature quan trọng
feature_importances = pd.Series(xgb_model.feature_importances_, index=X_train.columns)
feature_importances = feature_importances.sort_values(ascending=False)
print("Feature Importances:")
print(feature_importances.to_string())


**Note:**
- Đa số các feature đều ít ảnh hưởng đến biến mục tiêu, chỉ có một vài feature quan trọng. Nhưng các feature đó lại leak thông tin (ví dụ như review_count, review_to_sold_ratio, v.v...) nên cần loại bỏ trước khi fine-tune với top features.
- Ngoài ra em chỉ xóa các feature có ảnh hưởng là 0, không xóa hết dù các feature khác đều ít ảnh hưởng.

In [ ]:
# Xóa các cột gây nhiễu trước khi fine-tune với top features
cols_to_drop = ['review_count', 'review_to_sold_ratio', 'has_video', 'kw_sale', 'kw_siêu rẻ']
# Lấy top features nhưng phải trừ 2 cột này ra nếu nó lỡ nằm trong top
valid_features = [col for col in feature_importances.index if col not in cols_to_drop]

X_train_top = X_train[valid_features]
X_val_top = X_val[valid_features]
X_test_top = X_test[valid_features]

In [ ]:
# 3. Fine-tune hyperparameters với GridSearchCV hoặc RandomizedSearchCV
from sklearn.model_selection import RandomizedSearchCV
param_dist = {
    'n_estimators': [100, 500, 1000, 5000, 10000],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 5, 7, 9],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0]
}

random_search = RandomizedSearchCV(
    estimator=XGBRegressor(random_state=42),
    param_distributions=param_dist,
    n_iter=10,
    scoring='neg_mean_squared_error',
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train_top, y_train)
best_xgb_model = random_search.best_estimator_


In [ ]:
# 4. Train lại mô hình với các hyperparameters tốt nhất trên toàn bộ tập train (train + valid).
print("Đang huấn luyện model cuối cùng trên toàn bộ dữ liệu (Train + Val)...")

# Gộp dữ liệu lại (Nhớ chỉ gộp những feature Top đã chọn)
X_full_train = pd.concat([X_train_top, X_val_top])
y_full_train = pd.concat([y_train, y_val])

# Lấy tham số tốt nhất đã tìm được ở bước 3
best_params = random_search.best_params_

# Khởi tạo model cuối cùng với tham số đó
final_model = XGBRegressor(
    **best_params,        
    random_state=42,
    n_jobs=-1
)

# Fit trên toàn bộ dữ liệu
final_model.fit(X_full_train, y_full_train)

print("Đã train xong Final Model!")



In [ ]:
# 5. Đánh giá mô hình trên tập test với RMSE, MAE, R². Với biến mục tiêu `quantity_sold` đã log1p1, cần chuyển ngược về giá trị gốc trước khi tính các metric.
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
y_test_pred = final_model.predict(X_test_top)
y_test_pred_real = np.expm1(y_test_pred)
y_test_real = np.expm1(y_test)
rmse = np.sqrt(mean_squared_error(y_test_real, y_test_pred_real))
mae = mean_absolute_error(y_test_real, y_test_pred_real)
r2 = r2_score(y_test_real, y_test_pred_real)
print(f"Test RMSE: {rmse}")
print(f"Test MAE: {mae}")
print(f"Test R²: {r2}")

**Nhận định về model XGBoost:**

- Mô hình XGBoost có thể đạt hiệu suất khá tốt trên tập validation và test. Tuy nhiên, việc lựa chọn và loại bỏ các feature có ảnh hưởng thấp hoặc leak thông tin là rất quan trọng để tránh overfitting và cải thiện khả năng tổng quát hóa của mô hình.
- Với giá trị R² trên tập test khoảng 0.62, mô hình cho thấy khả năng giải thích biến động của biến mục tiêu là khá tốt, nhưng vẫn còn dư địa để cải thiện.